In [1]:
from ml_article_tagging.config import RAW_DATA_DIR

import pandas as pd
%load_ext autoreload
%autoreload 2

In [27]:
# Importing the data
raw_data_path = RAW_DATA_DIR / "dataset.csv"

df = pd.read_csv(raw_data_path)
df.head()

,id,created_on,title,description,tag
0,6,2020-02-20 06:43:18,Comparison between YOLO and RCNN on real world...,Bringing theory to experiment is cool. We can ...,computer-vision
1,7,2020-02-20 06:47:21,"Show, Infer & Tell: Contextual Inference for C...",The beauty of the work lies in the way it arch...,computer-vision
2,9,2020-02-24 16:24:45,Awesome Graph Classification,"A collection of important graph embedding, cla...",other
3,15,2020-02-28 23:55:26,Awesome Monte Carlo Tree Search,A curated list of Monte Carlo tree search pape...,other
4,25,2020-03-07 23:04:31,AttentionWalk,"A PyTorch Implementation of ""Watch Your Step: ...",other


### Feature Engineering
The title and description are combined into a single `text` feature.

In [28]:
df["text"] = df.title + " " + df.description

In [29]:
df.head()

,id,created_on,title,description,tag,text
0,6,2020-02-20 06:43:18,Comparison between YOLO and RCNN on real world...,Bringing theory to experiment is cool. We can ...,computer-vision,Comparison between YOLO and RCNN on real world...
1,7,2020-02-20 06:47:21,"Show, Infer & Tell: Contextual Inference for C...",The beauty of the work lies in the way it arch...,computer-vision,"Show, Infer & Tell: Contextual Inference for C..."
2,9,2020-02-24 16:24:45,Awesome Graph Classification,"A collection of important graph embedding, cla...",other,Awesome Graph Classification A collection of i...
3,15,2020-02-28 23:55:26,Awesome Monte Carlo Tree Search,A curated list of Monte Carlo tree search pape...,other,Awesome Monte Carlo Tree Search A curated list...
4,25,2020-03-07 23:04:31,AttentionWalk,"A PyTorch Implementation of ""Watch Your Step: ...",other,"AttentionWalk A PyTorch Implementation of ""Wat..."


### Cleaning
For transformer models, text preprocessing should be minimal. Transformers can understand punctuation, and removing stopwords may harm the contextual meaning of the sentence.

In [30]:
import re

In [31]:
def clean_text(text):

    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [32]:
# Apply to dataframe
original_df = df.copy()
df["text"] = df["text"].apply(clean_text)
print (f"{original_df.text.values[0]}\n{df.text.values[0]}")

Comparison between YOLO and RCNN on real world videos Bringing theory to experiment is cool. We can easily train models in colab and find the results in minutes.
comparison between yolo and rcnn on real world videos bringing theory to experiment is cool. we can easily train models in colab and find the results in minutes.


In [33]:
# DataFrame cleanup
df = df.drop(columns=["id", "created_on", "title", "description"])  # drop cols
df = df.dropna(subset=["tag"])  # drop nulls
df = df[["text", "tag"]] # rearrange cols
df.head()

,text,tag
0,comparison between yolo and rcnn on real world...,computer-vision
1,"show, infer & tell: contextual inference for c...",computer-vision
2,awesome graph classification a collection of i...,other
3,awesome monte carlo tree search a curated list...,other
4,"attentionwalk a pytorch implementation of ""wat...",other


### Encoding

In [41]:
# Label to index
tags = sorted(df.tag.unique())
print(tags)

class_to_index = {tag: i for i, tag in enumerate(tags)}
class_to_index

['computer-vision', 'mlops', 'natural-language-processing', 'other']


{'computer-vision': 0,
 'mlops': 1,
 'natural-language-processing': 2,
 'other': 3}

In [42]:
# Encode tags
df["tag"] = df["tag"].map(class_to_index)
df.head()

,text,tag
0,comparison between yolo and rcnn on real world...,0
1,"show, infer & tell: contextual inference for c...",0
2,awesome graph classification a collection of i...,3
3,awesome monte carlo tree search a curated list...,3
4,"attentionwalk a pytorch implementation of ""wat...",3


In [43]:
index_to_class = {v: k for k, v in class_to_index.items()}
index_to_class

{0: 'computer-vision',
 1: 'mlops',
 2: 'natural-language-processing',
 3: 'other'}

In [44]:
def decode(indices, index_to_class):
    return [index_to_class[index] for index in indices]

In [45]:
indices = [0, 1, 2, 1, 1, 0, 2]

decode(indices, index_to_class)

['computer-vision',
 'mlops',
 'natural-language-processing',
 'mlops',
 'mlops',
 'computer-vision',
 'natural-language-processing']

### Tokenize

In [11]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

text = "Transfer learning with transformers for text classification."

encoded = tokenizer([text], 
                           padding=False,
                           truncation=True,
                           max_length=512)

print ("input_ids:", encoded["input_ids"])
print ("attention_mask:", encoded["attention_mask"])
print (tokenizer.decode(encoded["input_ids"][0]))

input_ids: [[102, 2268, 1904, 190, 29155, 168, 3267, 2998, 205, 103]]
attention_mask: [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]
[CLS] transfer learning with transformers for text classification. [SEP]


In [12]:
encoded

{'input_ids': [[102, 2268, 1904, 190, 29155, 168, 3267, 2998, 205, 103]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}